In [0]:
T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscription_events")
T_customer = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_customer")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")

In [0]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

start_date = "2026-01-01"
end_date = "2026-03-31"
snapshot_date = (
    datetime.strptime(str(end_date), "%Y-%m-%d") - timedelta(days=0)
).strftime("%Y%m%d")

current_start = start_date
current_end = F.last_day(F.to_date(F.lit(start_date)))

prior_start = F.add_months(F.to_date(F.lit(start_date)), -9)
prior_end = F.date_sub(F.to_date(F.lit(start_date)), 1)

In [0]:
spark.range(1).select(
    F.lit(start_date).alias("current_start"),
    F.last_day(F.to_date(F.lit(start_date))).alias("current_end"),
    F.add_months(F.to_date(F.lit(start_date)), -9).alias("prior_start"),
    F.date_sub(F.to_date(F.lit(start_date)), 1).alias("prior_end")
).show(truncate=False)

In [0]:
T_active = (
    T_events
    .filter(
        (F.to_date("reporting_datetime").between(start_date, end_date)) &
        (F.col("event_type").isin("subscription_created", "subscription_renewed"))
    )
    .join(
        T_subscriber.filter(F.col("snapshot_date_id") == snapshot_date)
        .select("cde_id", "cc_user_profile_id"),
        on="cc_user_profile_id",
        how="left"
    )
    .groupBy("cde_id")
    .agg(F.countDistinct("invoice_id").alias("Invoices"))
    .filter(F.col("Invoices") == 3)
)

T_gift_plan = (
    T_events
    .filter(
        (F.col("event_type").isin("subscription_created", "subscription_renewed")) &
        (F.col("plan_id") == "gift-plan") &
        (
            F.to_date("reporting_datetime").between(current_start, current_end) |
            F.to_date("reporting_datetime").between(prior_start, prior_end)
        )
    )
    .join(
        T_subscriber.select("cc_user_profile_id", "cde_id"),
        on="cc_user_profile_id",
        how="left"
    )
    .select("cde_id")
    .distinct()
)


In [0]:
df_CC_Members = (
    T_subscriber
    .filter(F.col("snapshot_date_id") == snapshot_date)
    .filter(F.col("renewal_status").isin("Active", "Active_Non_Renewing", "Active_Annnual_Upgrade"))

    # --- Active members ---
    .join(T_active, on="cde_id", how="inner")
    .select(
        "cde_id",
        "cc_user_profile_id",
        "scene_membership_id",
        "plan_id",
        "renewal_status",
        "recognitions_left",
    )

    # --- Append gift members (USE EXISTING T_gift_cde) ---
    .unionByName(
        T_subscriber
        .filter(F.col("snapshot_date_id") == snapshot_date)
        .join(T_gift_plan, on="cde_id", how="inner")
        .select(
            "cde_id",
            "cc_user_profile_id",
            "scene_membership_id",
            "plan_id",
            "renewal_status",
            "recognitions_left",
        )
    )
    # --- Deduplicate ---
    .dropDuplicates(["cde_id"])
)

In [0]:
from pyspark.sql import functions as F

T_all_trans = (
    T_trans
    .join(
        T_location.select("LocationID", "LC_Location_Type_Description", "LC_Trade_Brand_Description"),
        T_trans.LocationId == T_location.LocationID,
        "left"
    )
    .join(
        T_date.select("dateid", "datefull"),
        T_trans.VisitDateId == T_date.dateid,
        "left"
    )
    .filter(F.col("datefull").between(start_date, end_date))
    .filter(F.col("CineClubFLAG") == "Yes")
    .filter(F.col("CineClub_Discount_Type_ID").isin(2, 3, 5, 6))  #2 for concessions, 3 for gaming, 5 for member priced, 6 for free
    .groupBy("CDE_ID", "datefull", T_trans.LocationId)
    .agg(
        # Concession Revenue
        F.sum(
            F.when((F.col("CineClub_Discount_Type_ID") == 2) & 
                   (F.col("LC_Location_Type_Description").isin("Cineplex Theatre")), F.col("Net_Revenue"))
        ).alias("Concession_Revenue"),

        # Gaming Revenue

        F.sum(
            F.when((F.col("CineClub_Discount_Type_ID") == 3) &
                (F.col("LC_Trade_Brand_Description").isin("Rec Room", "Playdium")), F.col("Net_Revenue"))
        ).alias("Gaming_Revenue"),

        # Ticket Revenue
        F.sum(
            F.when(
                (F.col("TicketFLAG") == "Yes") &
                (F.col("CineClub_Discount_Type_ID").isin(5, 6)) &
                (F.col("LC_Location_Type_Description") == "Cineplex Theatre"),
                F.col("Net_Revenue")
            )
        ).alias("Ticket_Revenue"),

        # Member tickets
        F.sum(
            F.when(
                (F.col("TicketFLAG") == "Yes") &
                (F.col("CineClub_Discount_Type_ID") == 5) & 
                (F.col("LC_Location_Type_Description").isin("Cineplex Theatre")),
                F.col("Quantity")
            )
        ).alias("Member_Priced_Tickets"),

        #Tickets Redeemed

        F.sum(
            F.when(
                (F.col("TicketFLAG") == "Yes") &
                (F.col("CineClub_Discount_Type_ID") == 6) & 
                (F.col("LC_Location_Type_Description").isin("Cineplex Theatre")),
                F.col("Quantity")
            )
        ).alias("Free_Tickets"),

        #Visits

    )
    .withColumn(
        "Visits",
        F.when(
            (F.col("Concession_Revenue") > 0) |
            (F.col("Ticket_Revenue") > 0) |
            (F.col("Gaming_Revenue") > 0),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
)

In [0]:
pip install openpyxl

In [0]:
import pandas as pd
df_pricing_pd = pd.read_excel("/Volumes/data_exploration_hub/marketing_hub/offline_data/Circuit Pricing.xlsx")
df_pricing = spark.createDataFrame(df_pricing_pd)

# Explode Theatre_ID into individual rows
df_pricing = df_pricing.withColumn(
    "Theatre_ID_Array",
    F.split(F.col("Theatre_ID"), ",")
)
df_price_exploded = (
    df_pricing
    .withColumn("Theatre_ID_exploded", F.explode("Theatre_ID_Array"))
    .withColumn("Theatre_ID_clean", F.col("Theatre_ID_exploded").cast("int"))
).drop("Category_Group", "Theatre_ID", "Theatre_ID_Array", "Theatre_ID_exploded", 
       "Child", "Senior", "Cineplex_Tuesday", "Tuesday")

In [0]:
from pyspark.sql import functions as F

T_savings = (
    T_all_trans
    # total tickets
    .withColumn(
        "No_of_tickets",
        F.coalesce(F.col("Member_Priced_Tickets"), F.lit(0)) +
        F.coalesce(F.col("Free_Tickets"), F.lit(0))
    )
    # join price table
    .join(
        df_price_exploded,
        T_all_trans.LocationId == df_price_exploded.Theatre_ID_clean,
        "left"
    )
    # ticket savings
    .withColumn(
        "Ticket_Savings",
        F.coalesce(
            F.round(
                (
                    (F.col("General") - F.lit(9.99)) * F.col("No_of_tickets")
                ) +
                (F.col("No_of_tickets") * F.lit(1.5)),
                2
            ),
            F.lit(0.0)
        )
    )
    # concession savings (25%)
    .withColumn(
        "Total_Savings_Concessions",
        F.coalesce(
            F.round(F.col("Concession_Revenue") * 0.25, 2),
            F.lit(0.0)
        )
    )
    # gaming savings (25%)
    .withColumn(
        "Total_Savings_Gaming",
        F.coalesce(
            F.round(F.col("Gaming_Revenue") * 0.25, 2),
            F.lit(0.0)
        )
    )
)

In [0]:
from pyspark.sql import functions as F

T_user_level = (
    T_savings
    .groupBy("CDE_ID")
    .agg(
        # Revenue
        F.sum("Concession_Revenue").alias("Total_Concession_Revenue"),
        F.sum("Ticket_Revenue").alias("Total_Ticket_Revenue"),
        F.sum("Gaming_Revenue").alias("Total_Gaming_Revenue"),

        # Tickets
        F.sum("Member_Priced_Tickets").alias("Total_Member_Priced_Tickets"),
        F.sum("Free_Tickets").alias("Total_Free_Tickets"),
        F.sum("No_of_tickets").alias("Total_Tickets"),

        # Visits
        F.sum("Visits").alias("Total_Visits"),

        # Savings
        F.sum("Ticket_Savings").alias("Total_Ticket_Savings"),
        F.sum("Total_Savings_Concessions").alias("Total_Concession_Savings"),
        F.sum("Total_Savings_Gaming").alias("Total_Gaming_Savings")
    )
)

In [0]:
df_final =(
    T_user_level
    .join(
        df_CC_Members,
        df_CC_Members.cde_id == T_user_level.CDE_ID,
        "inner"
    )
    .select(
        T_user_level.CDE_ID,
        "scene_membership_id",
        "plan_id",
        "renewal_status",
        "recognitions_left",
        "Total_Visits",
        "Total_Member_Priced_Tickets",
        "Total_Free_Tickets",
        "Total_Ticket_Revenue",
        "Total_Ticket_Savings",
        "Total_Concession_Revenue",
        "Total_Concession_Savings",
        "Total_Gaming_Revenue",
        "Total_Gaming_Savings"
).withColumnRenamed("recognitions_left", "Unused_Recognitions")
    .filter(F.col("Unused_recognitions")<100)
    .filter(F.col("Total_Visits")>0)
)

In [0]:
# df_final.display()

In [0]:
import pandas as pd

# 1) Read the CSV with only the columns you need
pdf_names = pd.read_excel(
    "/Volumes/data_exploration_hub/marketing_hub/offline_data/Subscriptions - March 2026 .xlsx",
    usecols=[
        "customers.id",
        "customers.first_name",
        "customers.last_name",
    ],
)

# 2) Ensure ID is string type for consistent joining
pdf_names["customers.id"] = pdf_names["customers.id"].astype("string")
pdf_names = pdf_names[pdf_names["customers.id"] != "AzqBunUAsPVB41nKT"]

# 3) Convert to Spark DataFrame
df_names = spark.createDataFrame(pdf_names)

# 4) Clean up column names and deduplicate
df_names = (
    df_names
    .select(
        F.col("`customers.id`").alias("cc_user_profile_id"),
        F.col("`customers.first_name`").alias("first_name"),
        F.col("`customers.last_name`").alias("last_name"),
    )
    .dropDuplicates(["cc_user_profile_id"])
)

# 5) Join to df_final
df_final = (
    T_user_level
    .join(
        df_CC_Members,
        df_CC_Members.cde_id == T_user_level.CDE_ID,
        "inner"
    )
    .join(
        df_names,
        df_CC_Members.cc_user_profile_id == df_names.cc_user_profile_id,
        "left"
    )
    .select(
        T_user_level.CDE_ID,
        "first_name",
        "last_name",
        "scene_membership_id",
        "plan_id",
        "renewal_status",
        "recognitions_left",
        "Total_Visits",
        "Total_Member_Priced_Tickets",
        "Total_Free_Tickets",
        # "Total_Ticket_Revenue",
        "Total_Ticket_Savings",
        # "Total_Concession_Revenue",
        "Total_Concession_Savings",
        # "Total_Gaming_Revenue",
        "Total_Gaming_Savings",
    )
    .withColumnRenamed("recognitions_left", "Unused_Recognitions")
    .withColumn("Total_Savings", F.col("Total_Ticket_Savings") + F.col("Total_Concession_Savings") + F.col("Total_Gaming_Savings"))
    .filter(F.col("renewal_status").like("%Active%"))
    .filter(F.col("Unused_Recognitions") < 100)
    .filter(F.col("Total_Visits") > 0)
    .filter(F.col("Total_Savings") >= 10)
)

In [0]:
df_final.display()